## Plot PPA Results (No Mutation)

This notebook reads the CSV outputs from the new no-mutation simulation notebook and produces:

1. a **correlation-based** contract map
2. an **expectation-based** level map
3. a **median-based** level map

The plotting table already contains the main statistics needed for these figures, including correlations, means, medians, regime indicators, and the selected contract outcome.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


## Parameter Setting

Update only this cell first.


In [ ]:
# ============================================================
# Main input / output
# ============================================================
# Folder written by the no-mutation simulation notebook.
# This folder is expected to be in the same workplace folder as this plotting notebook.
# With the default below, the notebook reads:
#   <workplace_dir>/Output files (Risk Neutral, No Mutation, Exact Global Search)/Simulation_Plot_Data_All_Matches.csv
results_dir = "Output files (Risk Neutral, No Mutation, Exact Global Search)"

# Optional explicit path to the plotting table.
# Leave as None to auto-use <workplace_dir>/<results_dir>/Simulation_Plot_Data_All_Matches.csv.
# If supplied, relative paths are resolved inside workplace_dir, not in a sibling ../result folder.
target_plot_data_file = None

# Folder that contains both this notebook and the output folder above.
# None means infer the notebook/current working folder. Set this only if your Jupyter working
# directory is different from the folder containing this notebook.
workplace_dir = None

# Folder for exported figures and filtered tables.
save_dir_name = "Figures and plot tables"

# ============================================================
# Source selection
# ============================================================
# Correlation source options:
#   "ref"       -> original preferred, fallback to simulated
#   "original"  -> historical match files only
#   "simulated" -> pooled baseline sample bank only
#   "target"    -> Code 2 shrunk target correlations (if present)
correlation_source = "ref"

# Level-stat source options:
#   "ref"       -> original preferred, fallback to simulated
#   "original"  -> historical match files only
#   "simulated" -> pooled baseline sample bank only
level_source = "ref"

# ============================================================
# Category thresholds
# ============================================================
shape_corr_low = -0.60
shape_corr_high = 0.30
basis_corr_low = -0.30
basis_corr_high = 0.30
level_balance_tolerance = 0.05

# ============================================================
# Plot filters
# ============================================================
filter_shape_regime = None          # 1 / 2 / 3 / None
filter_basis_regime = None          # 1 / 2 / 3 / None
filter_volume_mean_cat = None       # 1 / 2 / 3 / None
filter_price_mean_cat = None        # 1 / 2 / 3 / None
filter_volume_median_cat = None     # 1 / 2 / 3 / None
filter_price_median_cat = None      # 1 / 2 / 3 / None

filter_ppa_types = None             # e.g. ["Physical", "Virtual"]
filter_profile_types = None         # e.g. ["Fix", "AsG", "AsC", "Non-PPA"]
filter_match_ids = None             # e.g. [1, 2, 3]
filter_unusual_contracted_volume = None   # "Yes" / "No" / None

# ============================================================
# Plot style
# ============================================================
figsize = (11, 8)
point_size = 120
point_alpha = 0.90
edge_color = "black"
edge_width = 0.6

show_match_id_labels = False
label_font_size = 7

# Font-size controls
# - tick_font_size: x/y tick labels
# - axis_title_font_size: x/y axis titles
# - plot_title_font_size: optional plot title, if title is supplied
# - legend_font_size / legend_title_font_size: legend text and legend title
# - legend_marker_size: legend marker size only; plotted point size remains point_size
# Set any of these to None to let Matplotlib use its default.
tick_font_size = 16
axis_title_font_size = 16
plot_title_font_size = 16
legend_font_size = 16
legend_title_font_size = 16
legend_marker_size = 10

# Legend controls
# Only the profile-type legend is shown. The legend labels include point counts.
# If Non-PPA appears in the plotted data, it is included as a profile-type entry.
adaptive_legend = True                    # only show categories actually present in each plotted subset
legend_show_counts = True                 # show labels as, e.g., Fix (n=20)
legend_profile_loc = "lower right"

# Non-PPA controls
# These aliases are standardized to the canonical label below before filtering and plotting.
non_ppa_label = "Non-PPA"
non_ppa_aliases = [
    "Non-PPA", "Non PPA", "No PPA", "No-PPA",
    "No Contract", "NoContract", "None",
]

use_jitter = False
jitter_scale_x = 0.003
jitter_scale_y = 0.003
random_seed = 42

marker_map = {
    "Non-PPA": "x",
    "No Contract": "x",  # legacy alias; standardized to Non-PPA before plotting
    "Physical": "o",
}
color_map = {
    "Fix": "#1f77b4",
    "AsG": "#ff7f0e",
    "AsC": "#2ca02c",
    "Non-PPA": "#7f7f7f",
    "Unknown": "#d62728",
}

# ============================================================
# Axis labels and optional manual limits
# ============================================================
corr_x_axis_label = "Profile Shape Correlation"
corr_y_axis_label = "Nodal Price Correlation"
mean_x_axis_label = "Volume mismatch index (expectation)"
mean_y_axis_label = "Price spread index (expectation)"
median_x_axis_label = "Volume mismatch index (median)"
median_y_axis_label = "Price spread index (median)"

use_custom_corr_x_lim = False
use_custom_corr_y_lim = False
corr_x_axis_lim_custom = (-1.0, 1.0)
corr_y_axis_lim_custom = (-1.0, 1.0)

use_custom_mean_x_lim = False
use_custom_mean_y_lim = False
mean_x_axis_lim_custom = (-1.0, 1.0)
mean_y_axis_lim_custom = (-1.0, 1.0)

use_custom_median_x_lim = False
use_custom_median_y_lim = False
median_x_axis_lim_custom = (-1.0, 1.0)
median_y_axis_lim_custom = (-1.0, 1.0)

# ============================================================
# Export controls
# ============================================================
save_fig = True
export_filtered_plot_table = True
filtered_plot_table_name = "Filtered_Plot_Data.csv"

corr_fig_name = "Optimal_Contracts_Correlation_Scatter.png"
mean_fig_name = "Optimal_Contracts_Expectation_Level_Scatter.png"
median_fig_name = "Optimal_Contracts_Median_Level_Scatter.png"

SETTINGS = {
    "results_dir": results_dir,
    "target_plot_data_file": target_plot_data_file,
    "workplace_dir": workplace_dir,
    "save_dir_name": save_dir_name,
    "correlation_source": correlation_source,
    "level_source": level_source,
    "shape_corr_low": shape_corr_low,
    "shape_corr_high": shape_corr_high,
    "basis_corr_low": basis_corr_low,
    "basis_corr_high": basis_corr_high,
    "level_balance_tolerance": level_balance_tolerance,
    "filter_shape_regime": filter_shape_regime,
    "filter_basis_regime": filter_basis_regime,
    "filter_volume_mean_cat": filter_volume_mean_cat,
    "filter_price_mean_cat": filter_price_mean_cat,
    "filter_volume_median_cat": filter_volume_median_cat,
    "filter_price_median_cat": filter_price_median_cat,
    "filter_ppa_types": filter_ppa_types,
    "filter_profile_types": filter_profile_types,
    "filter_match_ids": filter_match_ids,
    "filter_unusual_contracted_volume": filter_unusual_contracted_volume,
    "figsize": figsize,
    "point_size": point_size,
    "point_alpha": point_alpha,
    "edge_color": edge_color,
    "edge_width": edge_width,
    "show_match_id_labels": show_match_id_labels,
    "label_font_size": label_font_size,
    "tick_font_size": tick_font_size,
    "axis_title_font_size": axis_title_font_size,
    "plot_title_font_size": plot_title_font_size,
    "legend_font_size": legend_font_size,
    "legend_title_font_size": legend_title_font_size,
    "legend_marker_size": legend_marker_size,
    "adaptive_legend": adaptive_legend,
    "legend_show_counts": legend_show_counts,
    "legend_profile_loc": legend_profile_loc,
    "non_ppa_label": non_ppa_label,
    "non_ppa_aliases": non_ppa_aliases,
    "use_jitter": use_jitter,
    "jitter_scale_x": jitter_scale_x,
    "jitter_scale_y": jitter_scale_y,
    "random_seed": random_seed,
    "marker_map": marker_map,
    "color_map": color_map,
    "corr_x_axis_label": corr_x_axis_label,
    "corr_y_axis_label": corr_y_axis_label,
    "mean_x_axis_label": mean_x_axis_label,
    "mean_y_axis_label": mean_y_axis_label,
    "median_x_axis_label": median_x_axis_label,
    "median_y_axis_label": median_y_axis_label,
    "corr_x_axis_lim": tuple(corr_x_axis_lim_custom) if use_custom_corr_x_lim else None,
    "corr_y_axis_lim": tuple(corr_y_axis_lim_custom) if use_custom_corr_y_lim else None,
    "mean_x_axis_lim": tuple(mean_x_axis_lim_custom) if use_custom_mean_x_lim else None,
    "mean_y_axis_lim": tuple(mean_y_axis_lim_custom) if use_custom_mean_y_lim else None,
    "median_x_axis_lim": tuple(median_x_axis_lim_custom) if use_custom_median_x_lim else None,
    "median_y_axis_lim": tuple(median_y_axis_lim_custom) if use_custom_median_y_lim else None,
    "save_fig": bool(save_fig),
    "export_filtered_plot_table": bool(export_filtered_plot_table),
    "filtered_plot_table_name": filtered_plot_table_name,
    "corr_fig_name": corr_fig_name,
    "mean_fig_name": mean_fig_name,
    "median_fig_name": median_fig_name,
}
SETTINGS


## Helper functions


In [ ]:
def infer_notebook_folder() -> Path:
    """
    Infer the folder that should be treated as the local workplace.

    In VS Code notebooks, __vsc_ipynb_file__ is usually available and points to the
    actual .ipynb file. Otherwise, this falls back to the current working directory.
    The resolver below is intentionally restricted to this folder and does not walk
    up to parent folders or search sibling result/results directories.
    """
    vsc_notebook_file = globals().get("__vsc_ipynb_file__")
    if vsc_notebook_file:
        return Path(vsc_notebook_file).expanduser().resolve().parent

    file_path = globals().get("__file__")
    if file_path:
        return Path(file_path).expanduser().resolve().parent

    return Path.cwd().resolve()


def resolve_workplace_dir(workplace_dir: Optional[str] = None) -> Path:
    """Resolve the folder containing this plotting notebook and its local output folders."""
    inferred = infer_notebook_folder()

    if workplace_dir is None or str(workplace_dir).strip() == "":
        return inferred

    p = Path(workplace_dir).expanduser()
    if not p.is_absolute():
        p = inferred / p
    return p.resolve()


PLOT_DATA_FILE_NAME = "Simulation_Plot_Data_All_Matches.csv"


def _unique_paths(paths) -> list[Path]:
    """Deduplicate candidate paths while preserving order."""
    out = []
    seen = set()
    for path in paths:
        if path is None:
            continue
        p = Path(path).expanduser()
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def first_existing(*paths, require_file: bool = False, require_dir: bool = False) -> Optional[Path]:
    for path in paths:
        if path is None:
            continue
        p = Path(path).expanduser()
        if not p.exists():
            continue
        if require_file and not p.is_file():
            continue
        if require_dir and not p.is_dir():
            continue
        return p.resolve()
    return None


def _candidate_result_dirs(results_dir: str, workplace_dir: Optional[str] = None) -> list[Path]:
    """
    Return local result-directory candidates.

    Relative results_dir values are resolved only as:
        <workplace_dir>/<results_dir>

    This avoids accidentally reading an older folder such as ../result/<results_dir>.
    """
    base_dir = resolve_workplace_dir(workplace_dir)
    results_candidate = Path(results_dir).expanduser()

    if results_candidate.is_absolute():
        candidates = [results_candidate]
    else:
        candidates = [base_dir / results_candidate]

    return _unique_paths(candidates)


def _candidate_manual_plot_paths(
    target_plot_data_file: str,
    result_dirs: list[Path],
    workplace_dir: Optional[str] = None,
) -> list[Path]:
    """Return local candidate locations for a manually specified plotting table."""
    base_dir = resolve_workplace_dir(workplace_dir)
    manual = Path(target_plot_data_file).expanduser()

    if manual.is_absolute():
        candidates = [manual]
    else:
        candidates = [base_dir / manual]
        for result_dir in result_dirs:
            candidates.append(result_dir / manual)
            # If manual is already a filename, this checks <result_dir>/<filename>.
            candidates.append(result_dir / manual.name)

    return _unique_paths(candidates)


def resolve_plot_data_path(
    results_dir: str,
    target_plot_data_file: Optional[str],
    workplace_dir: Optional[str] = None,
) -> Path:
    """
    Resolve the plotting table from the local workplace folder.

    The search is intentionally narrow:
      1. If target_plot_data_file is supplied, use it relative to workplace_dir unless absolute.
      2. Otherwise use <workplace_dir>/<results_dir>/Simulation_Plot_Data_All_Matches.csv.
      3. If results_dir itself is a CSV path, use it relative to workplace_dir unless absolute.

    It does not search parent folders, ../result, or ../results.
    """
    base_dir = resolve_workplace_dir(workplace_dir)
    result_dirs = _candidate_result_dirs(results_dir, workplace_dir)

    results_candidate = Path(results_dir).expanduser()
    if target_plot_data_file is None and results_candidate.suffix.lower() == ".csv":
        direct_file = results_candidate if results_candidate.is_absolute() else base_dir / results_candidate
        resolved = first_existing(direct_file, require_file=True)
        if resolved is not None:
            return resolved

    if target_plot_data_file is not None and str(target_plot_data_file).strip() != "":
        manual_candidates = _candidate_manual_plot_paths(target_plot_data_file, result_dirs, workplace_dir)
        resolved = first_existing(*manual_candidates, require_file=True)
        if resolved is not None:
            return resolved

        checked = "\n".join(f"  - {p}" for p in manual_candidates)
        raise FileNotFoundError(
            f"Could not resolve plotting table: {target_plot_data_file}\n"
            f"Workplace folder: {base_dir}\n"
            f"Checked only local candidate paths:\n{checked}"
        )

    auto_candidates = [result_dir / PLOT_DATA_FILE_NAME for result_dir in result_dirs]
    resolved = first_existing(*auto_candidates, require_file=True)
    if resolved is not None:
        return resolved

    checked = "\n".join(f"  - {p}" for p in auto_candidates)
    raise FileNotFoundError(
        f"Could not find {PLOT_DATA_FILE_NAME}.\n"
        f"Workplace folder: {base_dir}\n"
        f"Expected local path:\n"
        f"  {base_dir / results_dir / PLOT_DATA_FILE_NAME}\n\n"
        f"Checked only local candidate paths:\n{checked}\n\n"
        f"This notebook no longer searches ../result or ../results outside the workplace folder. "
        f"Move the output folder next to this notebook or set workplace_dir/target_plot_data_file explicitly."
    )


def _coerce_numeric(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        cleaned = value.replace(",", "").replace("$", "").replace("%", "").strip()
        if cleaned == "":
            return np.nan
        value = cleaned
    return pd.to_numeric(value, errors="coerce")


def choose_series(df: pd.DataFrame, column_candidates) -> pd.Series:
    for col in column_candidates:
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.nan, index=df.index, dtype=float)


def choose_text_series(df: pd.DataFrame, column_candidates, default: str = "Unknown") -> pd.Series:
    for col in column_candidates:
        if col in df.columns:
            return df[col].copy()
    return pd.Series(default, index=df.index, dtype="object")


def _compact_label(value) -> str:
    return "".join(ch for ch in str(value).lower() if ch.isalnum())


def _standardize_contract_label(value, settings: Dict, *, treat_missing_as_unknown: bool = True) -> str:
    if pd.isna(value):
        return "Unknown" if treat_missing_as_unknown else str(settings.get("non_ppa_label", "Non-PPA"))

    label = str(value).strip()
    if label == "" or label.lower() in {"nan", "<na>", "null"}:
        return "Unknown" if treat_missing_as_unknown else str(settings.get("non_ppa_label", "Non-PPA"))

    non_ppa_label = str(settings.get("non_ppa_label", "Non-PPA"))
    alias_values = settings.get("non_ppa_aliases", [non_ppa_label])
    alias_compact = {_compact_label(x) for x in alias_values}

    if _compact_label(label) in alias_compact:
        return non_ppa_label

    return label


def standardize_contract_columns(df: pd.DataFrame, settings: Dict) -> pd.DataFrame:
    """
    Standardize PPA and profile labels before filtering/plotting.

    This keeps Non-PPA rows in the plotted data when they exist and gives them an
    explicit marker/color rather than letting them fall into Unknown.
    """
    out = df.copy()

    ppa_raw = choose_text_series(
        out,
        [
            "ppa_type", "PPA type", "PPA Type",
            "contract_type", "contract", "selected_contract",
            "selected_ppa_type", "selected_contract_type", "optimal_ppa_type",
        ],
    )
    profile_raw = choose_text_series(
        out,
        [
            "profile_type", "Profile type", "Profile Type",
            "volume_profile", "selected_profile_type", "optimal_profile_type",
            "profile", "profile_choice",
        ],
    )

    ppa_series = ppa_raw.map(lambda x: _standardize_contract_label(x, settings))
    profile_series = profile_raw.map(lambda x: _standardize_contract_label(x, settings))

    non_ppa_label = str(settings.get("non_ppa_label", "Non-PPA"))

    # If one column explicitly says Non-PPA and the other is missing, treat the row as Non-PPA.
    ppa_series = ppa_series.mask(ppa_series.eq("Unknown") & profile_series.eq(non_ppa_label), non_ppa_label)
    profile_series = profile_series.mask(ppa_series.eq(non_ppa_label), non_ppa_label)

    out["ppa_type"] = ppa_series
    out["profile_type"] = profile_series

    return out


def classify_corr_regime_series(values: pd.Series, low: float, high: float,
                                label_high: str, label_mid: str, label_low: str) -> Tuple[pd.Series, pd.Series]:
    values = pd.to_numeric(values, errors="coerce")
    codes = pd.Series(np.nan, index=values.index, dtype=float)
    labels = pd.Series(pd.NA, index=values.index, dtype="object")

    codes.loc[values >= high] = 1
    labels.loc[values >= high] = label_high

    mid_mask = values.notna() & values.ge(low) & values.lt(high)
    codes.loc[mid_mask] = 2
    labels.loc[mid_mask] = label_mid

    low_mask = values.notna() & values.lt(low)
    codes.loc[low_mask] = 3
    labels.loc[low_mask] = label_low

    return codes, labels


def classify_level_series(values: pd.Series, tol: float,
                          label_low: str, label_mid: str, label_high: str) -> Tuple[pd.Series, pd.Series]:
    values = pd.to_numeric(values, errors="coerce")
    codes = pd.Series(np.nan, index=values.index, dtype=float)
    labels = pd.Series(pd.NA, index=values.index, dtype="object")

    low_mask = values.notna() & values.lt(-tol)
    mid_mask = values.notna() & values.ge(-tol) & values.le(tol)
    high_mask = values.notna() & values.gt(tol)

    codes.loc[low_mask] = 1
    labels.loc[low_mask] = label_low

    codes.loc[mid_mask] = 2
    labels.loc[mid_mask] = label_mid

    codes.loc[high_mask] = 3
    labels.loc[high_mask] = label_high

    return codes, labels


def signed_index(left, right, use_abs_denominator: bool = False):
    left = pd.to_numeric(left, errors="coerce")
    right = pd.to_numeric(right, errors="coerce")
    denominator = left.abs() + right.abs() if use_abs_denominator else left + right
    out = pd.Series(np.nan, index=left.index, dtype=float)
    valid = left.notna() & right.notna() & denominator.notna() & denominator.ne(0)
    out.loc[valid] = (left.loc[valid] - right.loc[valid]) / denominator.loc[valid]
    return out


def build_plot_fields(plot_df: pd.DataFrame, settings: Dict) -> pd.DataFrame:
    df = plot_df.copy()

    corr_source = str(settings["correlation_source"]).strip().lower()
    level_source = str(settings["level_source"]).strip().lower()

    corr_map = {
        "ref": ("shape_corr_ref", "basis_corr_ref"),
        "original": ("shape_corr_original", "basis_corr_original"),
        "simulated": ("shape_corr_simulated", "basis_corr_simulated"),
        "target": ("shape_corr_target", "basis_corr_target"),
    }
    level_map = {
        "ref": (
            "generation_mean_ref", "demand_mean_ref", "seller_lmp_mean_ref", "buyer_lmp_out_mean_ref",
            "generation_median_ref", "demand_median_ref", "seller_lmp_median_ref", "buyer_lmp_out_median_ref",
        ),
        "original": (
            "generation_original_mean", "demand_original_mean", "seller_lmp_original_mean", "buyer_lmp_out_original_mean",
            "generation_original_median", "demand_original_median", "seller_lmp_original_median", "buyer_lmp_out_original_median",
        ),
        "simulated": (
            "generation_simulated_mean", "demand_simulated_mean", "seller_lmp_simulated_mean", "buyer_lmp_out_simulated_mean",
            "generation_simulated_median", "demand_simulated_median", "seller_lmp_simulated_median", "buyer_lmp_out_simulated_median",
        ),
    }

    shape_col, basis_col = corr_map.get(corr_source, corr_map["ref"])
    (
        g_mean_col, d_mean_col, seller_price_mean_col, buyer_price_mean_col,
        g_median_col, d_median_col, seller_price_median_col, buyer_price_median_col,
    ) = level_map.get(level_source, level_map["ref"])

    df["shape_corr_plot"] = choose_series(df, [shape_col, "shape_corr_ref", "shape_corr_original", "shape_corr_simulated"])
    df["basis_corr_plot"] = choose_series(df, [basis_col, "basis_corr_ref", "basis_corr_original", "basis_corr_simulated"])

    df["seller_volume_mean_used"] = choose_series(df, [g_mean_col])
    df["buyer_volume_mean_used"] = choose_series(df, [d_mean_col])
    df["seller_price_mean_used"] = choose_series(df, [seller_price_mean_col])
    df["buyer_price_mean_used"] = choose_series(df, [buyer_price_mean_col])

    df["seller_volume_median_used"] = choose_series(df, [g_median_col])
    df["buyer_volume_median_used"] = choose_series(df, [d_median_col])
    df["seller_price_median_used"] = choose_series(df, [seller_price_median_col])
    df["buyer_price_median_used"] = choose_series(df, [buyer_price_median_col])

    df["vol_mismatch_mean_plot"] = signed_index(
        df["seller_volume_mean_used"], df["buyer_volume_mean_used"], use_abs_denominator=False
    )
    df["price_spread_mean_plot"] = signed_index(
        df["seller_price_mean_used"], df["buyer_price_mean_used"], use_abs_denominator=True
    )
    df["vol_mismatch_median_plot"] = signed_index(
        df["seller_volume_median_used"], df["buyer_volume_median_used"], use_abs_denominator=False
    )
    df["price_spread_median_plot"] = signed_index(
        df["seller_price_median_used"], df["buyer_price_median_used"], use_abs_denominator=True
    )

    df["shape_regime"], df["shape_regime_label"] = classify_corr_regime_series(
        df["shape_corr_plot"],
        settings["shape_corr_low"],
        settings["shape_corr_high"],
        "Coincident profile",
        "Weakly coupled profile",
        "Mismatched profile",
    )
    df["basis_regime"], df["basis_regime_label"] = classify_corr_regime_series(
        df["basis_corr_plot"],
        settings["basis_corr_low"],
        settings["basis_corr_high"],
        "Convergent market",
        "Moderately divergent market",
        "Highly divergent market",
    )
    df["volume_mean_cat"], df["volume_mean_cat_label"] = classify_level_series(
        df["vol_mismatch_mean_plot"],
        settings["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )
    df["price_mean_cat"], df["price_mean_cat_label"] = classify_level_series(
        df["price_spread_mean_plot"],
        settings["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )
    df["volume_median_cat"], df["volume_median_cat_label"] = classify_level_series(
        df["vol_mismatch_median_plot"],
        settings["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )
    df["price_median_cat"], df["price_median_cat_label"] = classify_level_series(
        df["price_spread_median_plot"],
        settings["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )

    df["combined_category"] = (
        "S" + df["shape_regime"].astype("Int64").astype(str) +
        "_B" + df["basis_regime"].astype("Int64").astype(str) +
        "_Vm" + df["volume_mean_cat"].astype("Int64").astype(str) +
        "_Pm" + df["price_mean_cat"].astype("Int64").astype(str) +
        "_Vd" + df["volume_median_cat"].astype("Int64").astype(str) +
        "_Pd" + df["price_median_cat"].astype("Int64").astype(str)
    )

    df = standardize_contract_columns(df, settings)

    return df


def apply_filters(plot_df: pd.DataFrame, settings: Dict) -> pd.DataFrame:
    df = plot_df.copy()

    filter_map = {
        "shape_regime": settings["filter_shape_regime"],
        "basis_regime": settings["filter_basis_regime"],
        "volume_mean_cat": settings["filter_volume_mean_cat"],
        "price_mean_cat": settings["filter_price_mean_cat"],
        "volume_median_cat": settings["filter_volume_median_cat"],
        "price_median_cat": settings["filter_price_median_cat"],
    }
    for col, value in filter_map.items():
        if value is not None:
            df = df.loc[pd.to_numeric(df[col], errors="coerce") == float(value)].copy()

    if settings["filter_ppa_types"] is not None:
        allowed = {_standardize_contract_label(x, settings) for x in settings["filter_ppa_types"]}
        df = df.loc[df["ppa_type"].astype(str).isin(allowed)].copy()

    if settings["filter_profile_types"] is not None:
        allowed = {_standardize_contract_label(x, settings) for x in settings["filter_profile_types"]}
        df = df.loc[df["profile_type"].astype(str).isin(allowed)].copy()

    if settings["filter_match_ids"] is not None:
        allowed = {int(x) for x in settings["filter_match_ids"]}
        df = df.loc[pd.to_numeric(df["match_id"], errors="coerce").astype("Int64").isin(allowed)].copy()

    if settings["filter_unusual_contracted_volume"] is not None and "unusual_contracted_volume" in df.columns:
        df = df.loc[df["unusual_contracted_volume"].astype(str) == str(settings["filter_unusual_contracted_volume"])].copy()

    return df.reset_index(drop=True)


def format_active_filters(settings: Dict) -> Dict[str, object]:
    keys = [
        "filter_shape_regime", "filter_basis_regime", "filter_volume_mean_cat", "filter_price_mean_cat",
        "filter_volume_median_cat", "filter_price_median_cat", "filter_ppa_types",
        "filter_profile_types", "filter_match_ids", "filter_unusual_contracted_volume",
    ]
    return {key: settings[key] for key in keys if settings.get(key) is not None}


def _ordered_legend_labels(present_labels, preferred_order: Dict) -> list:
    """Return present labels in the preferred map order, then any extras alphabetically."""
    present = {str(label).strip() for label in present_labels if str(label).strip() not in {"", "nan", "<NA>", "None"}}
    ordered = [label for label in preferred_order.keys() if label in present]
    extras = sorted(label for label in present if label not in preferred_order)
    return ordered + extras


def make_contract_scatter(df: pd.DataFrame, x_col: str, y_col: str,
                          x_label: str, y_label: str, fig_name: str,
                          title: str, settings: Dict, x_lim=None, y_lim=None,
                          save_dir: Optional[Path] = None) -> pd.DataFrame:
    plot_subset = df.loc[df[x_col].notna() & df[y_col].notna()].copy()
    print(f"{fig_name}: {len(plot_subset)} points")

    if plot_subset.empty:
        print("No rows remain after filtering for the requested plot.")
        return plot_subset

    rng = np.random.default_rng(settings["random_seed"])
    x_vals = plot_subset[x_col].to_numpy(dtype=float)
    y_vals = plot_subset[y_col].to_numpy(dtype=float)

    if settings["use_jitter"]:
        x_vals = x_vals + rng.normal(0.0, settings["jitter_scale_x"], size=len(plot_subset))
        y_vals = y_vals + rng.normal(0.0, settings["jitter_scale_y"], size=len(plot_subset))

    marker_map = settings.get("marker_map", {})
    color_map = settings.get("color_map", {})
    unknown_color = color_map.get("Unknown", "#d62728")

    # Normalize plotted categories once. The same normalized values drive both plotting and legends.
    plot_subset = standardize_contract_columns(plot_subset, settings)
    ppa_series = plot_subset["ppa_type"]
    profile_series = plot_subset["profile_type"]

    fig, ax = plt.subplots(figsize=settings["figsize"])

    non_ppa_label = str(settings.get("non_ppa_label", "Non-PPA"))
    non_ppa_color = color_map.get(non_ppa_label, "#7f7f7f")

    for idx, (_, row) in enumerate(plot_subset.iterrows()):
        ppa_type = ppa_series.iloc[idx]
        profile_type = profile_series.iloc[idx]
        is_non_ppa = (ppa_type == non_ppa_label) or (profile_type == non_ppa_label)

        if is_non_ppa:
            marker = marker_map.get(non_ppa_label, "x")
            color = non_ppa_color
            edgecolors = non_ppa_color
            linewidths = max(float(settings.get("edge_width", 0.6)), 1.4)
        else:
            marker = marker_map.get(ppa_type, "o")
            color = color_map.get(profile_type, unknown_color)
            edgecolors = settings["edge_color"]
            linewidths = settings["edge_width"]

        scatter_kwargs = {
            "s": settings["point_size"],
            "alpha": settings["point_alpha"],
            "marker": marker,
            "color": color,
            "linewidths": linewidths,
        }
        if not is_non_ppa:
            scatter_kwargs["edgecolors"] = edgecolors

        ax.scatter(
            x_vals[idx],
            y_vals[idx],
            **scatter_kwargs,
        )

        if settings["show_match_id_labels"]:
            ax.text(
                x_vals[idx],
                y_vals[idx],
                str(int(row["match_id"])),
                fontsize=settings["label_font_size"],
                alpha=0.80,
            )

    ax.axhline(0.0, linewidth=1.0)
    ax.axvline(0.0, linewidth=1.0)
    if x_lim is not None:
        ax.set_xlim(x_lim)
    if y_lim is not None:
        ax.set_ylim(y_lim)

    axis_title_font_size = settings.get("axis_title_font_size")
    tick_font_size = settings.get("tick_font_size")
    plot_title_font_size = settings.get("plot_title_font_size")

    ax.set_xlabel(x_label, fontsize=axis_title_font_size)
    ax.set_ylabel(y_label, fontsize=axis_title_font_size)
    if tick_font_size is not None:
        ax.tick_params(axis="both", which="both", labelsize=tick_font_size)
    if title:
        ax.set_title(title, fontsize=plot_title_font_size)

    adaptive_legend = bool(settings.get("adaptive_legend", True))
    legend_marker_size = settings.get("legend_marker_size", 8)
    legend_font_size = settings.get("legend_font_size")
    legend_title_font_size = settings.get("legend_title_font_size")
    legend_show_counts = bool(settings.get("legend_show_counts", True))

    if adaptive_legend:
        profile_legend_labels = _ordered_legend_labels(set(profile_series), color_map)
    else:
        # Keep the historical non-adaptive behavior of hiding Unknown unless it is genuinely plotted.
        profile_legend_labels = [label for label in color_map.keys() if label != "Unknown"]

    profile_counts = profile_series.value_counts(dropna=False).to_dict()

    def _legend_label(label: str) -> str:
        count = int(profile_counts.get(label, 0))
        return f"{label} ({count})" if legend_show_counts else label

    color_handles = []
    for label in profile_legend_labels:
        profile_color = color_map.get(label, unknown_color)
        is_non_ppa_label = label == non_ppa_label
        profile_marker = marker_map.get(non_ppa_label, "x") if is_non_ppa_label else "o"
        marker_edge_color = profile_color if is_non_ppa_label else settings["edge_color"]
        marker_face_color = "none" if is_non_ppa_label else profile_color
        marker_edge_width = max(float(settings.get("edge_width", 0.6)), 1.4) if is_non_ppa_label else settings["edge_width"]

        color_handles.append(
            Line2D(
                [0], [0],
                marker=profile_marker,
                linestyle="",
                markersize=legend_marker_size,
                markerfacecolor=marker_face_color,
                markeredgecolor=marker_edge_color,
                markeredgewidth=marker_edge_width,
                label=_legend_label(label),
            )
        )

    if color_handles:
        ax.legend(
            handles=color_handles,
            title="Profile Type",
            loc=settings.get("legend_profile_loc", "lower left"),
            fontsize=legend_font_size,
            title_fontsize=legend_title_font_size,
        )

    fig.tight_layout()

    if settings["save_fig"] and save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        fig_path = save_dir / fig_name
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure: {fig_path}")

    plt.show()
    return plot_subset

def summarize_category_levels(plot_df: pd.DataFrame) -> Dict[str, list]:
    cols = [
        "shape_regime", "basis_regime", "volume_mean_cat",
        "price_mean_cat", "volume_median_cat", "price_median_cat",
    ]
    out = {}
    for col in cols:
        if col in plot_df.columns:
            vals = (
                pd.to_numeric(plot_df[col], errors="coerce")
                .dropna()
                .astype(int)
                .sort_values()
                .unique()
                .tolist()
            )
            out[col] = vals
    return out

## Load the plotting table


In [ ]:
plot_data_path = resolve_plot_data_path(
    results_dir=SETTINGS["results_dir"],
    target_plot_data_file=SETTINGS["target_plot_data_file"],
    workplace_dir=SETTINGS["workplace_dir"],
)
# Save figures next to the resolved local plotting table.
results_dir_path = plot_data_path.parent.resolve()
save_dir = (results_dir_path / SETTINGS["save_dir_name"]).resolve()
save_dir.mkdir(parents=True, exist_ok=True)

plot_df_raw = read_csv_optimized(plot_data_path)
plot_df_full = build_plot_fields(plot_df_raw, SETTINGS)
plot_df_plot = apply_filters(plot_df_full, SETTINGS)

print("Resolved paths:")
print("  Plot-data table :", plot_data_path)
print("  Save directory  :", save_dir)
print()
print("Plot-data rows before filtering:", len(plot_df_full))
print("Plot-data rows after filtering :", len(plot_df_plot))
print("Active filters:", format_active_filters(SETTINGS))

if SETTINGS["export_filtered_plot_table"]:
    export_path = save_dir / SETTINGS["filtered_plot_table_name"]
    plot_df_plot.to_csv(export_path, index=False)
    print(f"Filtered plot-data table exported to: {export_path}")

plot_df_plot.head()


## Create the scatter plots


In [ ]:
corr_points = make_contract_scatter(
    plot_df_plot,
    x_col="shape_corr_plot",
    y_col="basis_corr_plot",
    x_label=SETTINGS["corr_x_axis_label"],
    y_label=SETTINGS["corr_y_axis_label"],
    fig_name=SETTINGS["corr_fig_name"],
    title="",
    settings=SETTINGS,
    x_lim=SETTINGS["corr_x_axis_lim"],
    y_lim=SETTINGS["corr_y_axis_lim"],
    save_dir=save_dir,
)

mean_points = make_contract_scatter(
    plot_df_plot,
    x_col="vol_mismatch_mean_plot",
    y_col="price_spread_mean_plot",
    x_label=SETTINGS["mean_x_axis_label"],
    y_label=SETTINGS["mean_y_axis_label"],
    fig_name=SETTINGS["mean_fig_name"],
    title="",
    settings=SETTINGS,
    x_lim=SETTINGS["mean_x_axis_lim"],
    y_lim=SETTINGS["mean_y_axis_lim"],
    save_dir=save_dir,
)

median_points = make_contract_scatter(
    plot_df_plot,
    x_col="vol_mismatch_median_plot",
    y_col="price_spread_median_plot",
    x_label=SETTINGS["median_x_axis_label"],
    y_label=SETTINGS["median_y_axis_label"],
    fig_name=SETTINGS["median_fig_name"],
    title="",
    settings=SETTINGS,
    x_lim=SETTINGS["median_x_axis_lim"],
    y_lim=SETTINGS["median_y_axis_lim"],
    save_dir=save_dir,
)


## Quick summaries


In [ ]:
print("Available category levels in the full plotting table:")
for col, levels in summarize_category_levels(plot_df_full).items():
    print(f"  {col}: {levels if levels else 'None'}")

print("\nAvailable category levels in the filtered plotting table:")
for col, levels in summarize_category_levels(plot_df_plot).items():
    print(f"  {col}: {levels if levels else 'None'}")

print("\nPPA type counts in the filtered plotting table:")
if "ppa_type" in plot_df_plot.columns:
    print(plot_df_plot["ppa_type"].value_counts(dropna=False))

print("\nProfile-type counts in the filtered plotting table:")
if "profile_type" in plot_df_plot.columns:
    print(plot_df_plot["profile_type"].value_counts(dropna=False))

debug_cols = [
    "match_id", "combined_category",
    "shape_regime", "basis_regime",
    "volume_mean_cat", "price_mean_cat",
    "volume_median_cat", "price_median_cat",
    "shape_corr_plot", "basis_corr_plot",
    "vol_mismatch_mean_plot", "price_spread_mean_plot",
    "vol_mismatch_median_plot", "price_spread_median_plot",
    "ppa_type", "profile_type", "volume_mw", "strike_price_mwh",
]
debug_cols = [col for col in debug_cols if col in plot_df_plot.columns]

print("\nFirst few filtered rows:")
print(plot_df_plot[debug_cols].head(10).to_string(index=False))
